# Background and motivation
So one of the things we've been discussing in this unit is how *the internet* works, via a series of protocols built on top of a loose and every-changing network. 

We talked about the idea of the "application layer" at the top of the stack of protocols, the part where the actual things we use all the time live.

One of the most important "applications" in the internet is HTTP: HTTP, as you may now, is the HyperText Transfer Protocol. The thing that makes the entirety of the web work!

You may, if you're significantly younger than me, find the term "the web" a little anachronous but I use it intentionally because I want to point out that the internet is actually a lot more than web sites and web servers! The web just kind of ate almost every user-facing part of the internet and now it's hard to tell that there ever could have been anything else!

But there are, though, other protocols that are still used like Gopher or Gemini (insert links later) that are alternative to web sites as well as protocols that are 

The fun thing about all of these protocols is that they're "plain text" protocols, in other words they're things that do all their work just in plane text that you can read!



In [11]:
import socket
import ssl
from IPython.display import HTML, display
import html

host = "example.com"
port = 443

request = (
    "GET / HTTP/1.1\r\n"
    f"Host: {host}\r\n"
    "Connection: close\r\n"
    "Accept-Encoding: identity\r\n"
    "\r\n"
)

context = ssl.create_default_context()

def decode_chunked_body(body):
    decoded = bytearray()
    position = 0

    while True:
        line_end = body.find(b"\r\n", position)
        if line_end == -1:
            raise ValueError("Malformed chunked body: missing chunk-size line ending")

        size_line = body[position:line_end]

        # Chunk size may have extensions after semicolon.
        # Example: b"22f;some-extension"
        size_text = size_line.split(b";", 1)[0]
        chunk_size = int(size_text, 16)

        position = line_end + 2

        if chunk_size == 0:
            break

        decoded.extend(body[position:position + chunk_size])
        position += chunk_size + 2  # skip chunk data plus trailing CRLF

    return bytes(decoded)

with socket.create_connection((host, port), timeout=10) as raw_sock:
    with context.wrap_socket(raw_sock, server_hostname=host) as sock:
        sock.sendall(request.encode("ascii"))

        response = b""
        while True:
            chunk = sock.recv(4096)
            if not chunk:
                break
            response += chunk

headerraw, bodyraw = response.split(b"\r\n\r\n", 1)
headers = headerraw.decode("latin-1")


if "transfer-encoding: chunked" in headers.lower():
    bodyraw = decode_chunked_body(bodyraw)

body = bodyraw.decode("utf-8", errors="replace")

print("HEADERS")
print(headers)
print("HTML")
print(body)

display(HTML(f"""
<iframe
    srcdoc="{html.escape(body)}"
    width="100%"
    height="300"
    style="border: 1px solid #ccc;">
</iframe>
"""))

HEADERS
HTTP/1.1 200 OK
Date: Fri, 19 Jun 2026 22:25:14 GMT
Content-Type: text/html
Transfer-Encoding: chunked
Connection: close
Server: cloudflare
last-modified: Fri, 19 Jun 2026 20:01:26 GMT
allow: GET, HEAD
Accept-Ranges: bytes
Age: 5163
cf-cache-status: HIT
CF-RAY: a0e5f2d2ce8245f7-SEA
HTML
<!doctype html><html lang="en"><head><title>Example Domain</title><link rel="icon" href="data:,"><meta name="viewport" content="width=device-width, initial-scale=1"><style>body{background:#eee;width:60vw;margin:15vh auto;font-family:system-ui,sans-serif}h1{font-size:1.5em}div{opacity:0.8}a:link,a:visited{color:#348}</style></head><body><div><h1>Example Domain</h1><p>This domain is for use in documentation examples without needing permission. Avoid use in operations.</p><p><a href="https://iana.org/domains/example">Learn more</a></p></div></body></html>

